# Comprehensive Vector Database Test

Test all Qdrant database functionality including:
- Single-vector and multi-vector modes
- Distance metrics (COSINE, DOT, MANHATTAN, EUCLID)
- HNSW index parameters
- Search-time parameters

In [1]:
# Install packages
import subprocess
import sys

packages = ['qdrant-client', 'numpy']
for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print("✓ Packages installed")

✓ Packages installed


In [ ]:
# Import the vector database module
import sys
import numpy as np

sys.path.insert(0, '..')
from indexing.vector_database import QdrantConfig, QdrantVectorDB

print("✓ Imports successful")

✓ Imports successful


In [3]:
# Test 1: Create config and database
print("Test 1: Initialize Database")
print("=" * 50)

config = QdrantConfig(
    db_mode="memory",  # Fast in-memory for testing
    collection_name="test_collection",
    embedding_dimension=384
)

vdb = QdrantVectorDB(config)
vdb.create_collection(force_recreate=True)

print(f"✓ Database initialized in {config.db_mode} mode")
print(f"✓ Collection '{config.collection_name}' created\n")

Test 1: Initialize Database
Initializing in-memory Qdrant...
✓ In-memory Qdrant initialized
✓ Collection 'test_collection' created
  Mode: Single-vector (384D, COSINE)
✓ Database initialized in memory mode
✓ Collection 'test_collection' created



In [4]:
# Test 2: Index some documents
print("Test 2: Index Documents")
print("=" * 50)

# Create some fake embeddings and documents
documents = []
for i in range(5):
    embedding = np.random.randn(384)  # 384-dim random vectors
    doc = {
        'id': i,
        'embedding': embedding,
        'text': f'Sample document {i}',
        'metadata': {'source': 'test', 'doc_num': i}
    }
    documents.append(doc)

num_indexed = vdb.index_documents(documents)
print(f"✓ Indexed {num_indexed} documents")
print(f"✓ Total docs in collection: {vdb.count_documents()}\n")

Test 2: Index Documents
✓ Total 5 documents indexed
✓ Indexed 5 documents
✓ Total docs in collection: 5



In [5]:
# Test 3: Retrieve similar documents
print("Test 3: Retrieve Documents")
print("=" * 50)

# Use the first document's embedding as query
query_embedding = documents[0]['embedding']

results = vdb.retrieve(query_embedding, top_k=3)
print(f"\nQuery with embedding from '{documents[0]['text']}'")
print(f"Retrieved {len(results)} results:\n")

for i, doc in enumerate(results, 1):
    print(f"{i}. ID: {doc['id']}")
    print(f"   Text: {doc['text']}")
    print(f"   Score: {doc['score']:.4f}")
    print(f"   Metadata: {doc['payload']}")
    print()

Test 3: Retrieve Documents

Query with embedding from 'Sample document 0'
Retrieved 3 results:

1. ID: 0
   Text: Sample document 0
   Score: 1.0000
   Metadata: {'text': 'Sample document 0', 'source': 'test', 'doc_num': 0}

2. ID: 4
   Text: Sample document 4
   Score: 0.0804
   Metadata: {'text': 'Sample document 4', 'source': 'test', 'doc_num': 4}

3. ID: 1
   Text: Sample document 1
   Score: 0.0062
   Metadata: {'text': 'Sample document 1', 'source': 'test', 'doc_num': 1}



In [6]:
# Test 4: Retrieve by specific IDs
print("Test 4: Retrieve by ID")
print("=" * 50)

doc_by_ids = vdb.retrieve_by_ids([1, 3])
print(f"Retrieved documents with IDs [1, 3]:\n")

for doc in doc_by_ids:
    print(f"ID: {doc['id']}")
    print(f"Text: {doc['text']}")
    print(f"Metadata: {doc['payload']}")
    print()

Test 4: Retrieve by ID
Retrieved documents with IDs [1, 3]:

ID: 1
Text: Sample document 1
Metadata: {'text': 'Sample document 1', 'source': 'test', 'doc_num': 1}

ID: 3
Text: Sample document 3
Metadata: {'text': 'Sample document 3', 'source': 'test', 'doc_num': 3}



In [7]:
# Test 5: Collection info
print("Test 5: Collection Info")
print("=" * 50)

info = vdb.get_collection_info()
print(f"Collection: {config.collection_name}")
print(f"Total documents: {vdb.count_documents()}")
print(f"Vector dimension: {config.embedding_dimension}")
print(f"\nFull info: {info}\n")

Test 5: Collection Info
Collection: test_collection
Total documents: 5
Vector dimension: 384

Full info: status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=0 points_count=5 segments_count=1 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=384, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=None, sharding_method=None, replication_factor=None, write_consistency_factor=None, read_fan_out_factor=None, on_disk_payload=None, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=None, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=20000, flush_interval_

In [8]:
# Summary
print("\n" + "="*50)
print("ALL TESTS PASSED ✓")
print("="*50)
print("\nThe vector database module works correctly:")
print("✓ Database initialization")
print("✓ Collection creation")
print("✓ Document indexing")
print("✓ Similarity search")
print("✓ ID-based retrieval")
print("✓ Collection info retrieval")


ALL TESTS PASSED ✓

The vector database module works correctly:
✓ Database initialization
✓ Collection creation
✓ Document indexing
✓ Similarity search
✓ ID-based retrieval
✓ Collection info retrieval


## Advanced Tests - Distance Metrics and HNSW

Test different distance metrics and index parameters

In [9]:
# Test 6: Different distance metrics
print("\nTest 6: Different Distance Metrics")
print("=" * 50)

distance_metrics = ["COSINE", "DOT", "MANHATTAN", "EUCLID"]

for metric in distance_metrics:
    print(f"\nTesting {metric} distance metric...")
    
    config_metric = QdrantConfig(
        db_mode="memory",
        collection_name=f"test_{metric.lower()}",
        embedding_dimension=384,
        distance=metric
    )
    
    vdb_metric = QdrantVectorDB(config_metric)
    vdb_metric.create_collection(force_recreate=True)
    
    # Create test documents
    test_docs = []
    for i in range(3):
        test_docs.append({
            'id': i,
            'embedding': np.random.randn(384),
            'text': f'Document {i} ({metric})',
            'metadata': {'metric': metric}
        })
    
    vdb_metric.index_documents(test_docs)
    
    # Test retrieval
    query_emb = test_docs[0]['embedding']
    results = vdb_metric.retrieve(query_emb, top_k=2)
    
    print(f"  ✓ {metric} retrieval successful: {len(results)} results")

print("\n✓ All distance metrics work correctly")


Test 6: Different Distance Metrics

Testing COSINE distance metric...
Initializing in-memory Qdrant...
✓ In-memory Qdrant initialized
✓ Collection 'test_cosine' created
  Mode: Single-vector (384D, COSINE)
✓ Total 3 documents indexed
  ✓ COSINE retrieval successful: 2 results

Testing DOT distance metric...
Initializing in-memory Qdrant...
✓ In-memory Qdrant initialized
✓ Collection 'test_dot' created
  Mode: Single-vector (384D, DOT)
✓ Total 3 documents indexed
  ✓ DOT retrieval successful: 2 results

Testing MANHATTAN distance metric...
Initializing in-memory Qdrant...
✓ In-memory Qdrant initialized
✓ Collection 'test_manhattan' created
  Mode: Single-vector (384D, MANHATTAN)
✓ Total 3 documents indexed
  ✓ MANHATTAN retrieval successful: 2 results

Testing EUCLID distance metric...
Initializing in-memory Qdrant...
✓ In-memory Qdrant initialized
✓ Collection 'test_euclid' created
  Mode: Single-vector (384D, EUCLID)
✓ Total 3 documents indexed
  ✓ EUCLID retrieval successful: 2 resu

In [10]:
# Test 7: HNSW Index Parameters
print("\nTest 7: HNSW Index Parameters")
print("=" * 50)

config_hnsw = QdrantConfig(
    db_mode="memory",
    collection_name="test_hnsw",
    embedding_dimension=384,
    distance="COSINE",
    hnsw_config={
        "m": 16,           # Number of connections per element
        "ef_construct": 200,  # Size of the dynamic list
        "full_scan_threshold": 10000  # Full scan threshold
    }
)

vdb_hnsw = QdrantVectorDB(config_hnsw)
vdb_hnsw.create_collection(force_recreate=True)

# Index documents
hnsw_docs = []
for i in range(5):
    hnsw_docs.append({
        'id': i,
        'embedding': np.random.randn(384),
        'text': f'HNSW test document {i}',
        'metadata': {'origin': 'hnsw_test'}
    })

vdb_hnsw.index_documents(hnsw_docs)

# Test retrieval
query_emb = hnsw_docs[0]['embedding']
results = vdb_hnsw.retrieve(query_emb, top_k=3)

print(f"✓ HNSW configuration applied successfully")
print(f"✓ Retrieved {len(results)} documents with HNSW index")


Test 7: HNSW Index Parameters
Initializing in-memory Qdrant...
✓ In-memory Qdrant initialized
✓ Collection 'test_hnsw' created
  Mode: Single-vector (384D, COSINE)
✓ Total 5 documents indexed
✓ HNSW configuration applied successfully
✓ Retrieved 3 documents with HNSW index


In [11]:
# Test 8: Search-Time Parameters
print("\nTest 8: Search-Time Parameters")
print("=" * 50)

config_search = QdrantConfig(
    db_mode="memory",
    collection_name="test_search_params",
    embedding_dimension=384,
    distance="COSINE",
    search_params={"ef": 100}  # Search-time parameter for HNSW
)

vdb_search = QdrantVectorDB(config_search)
vdb_search.create_collection(force_recreate=True)

# Index documents
search_docs = []
for i in range(10):
    search_docs.append({
        'id': i,
        'embedding': np.random.randn(384),
        'text': f'Search param test document {i}',
        'metadata': {'batch': i // 2}
    })

vdb_search.index_documents(search_docs)

# Test retrieval with search parameters
query_emb = search_docs[0]['embedding']
results = vdb_search.retrieve(query_emb, top_k=5)

print(f"✓ Search parameters applied successfully")
print(f"✓ Retrieved {len(results)} documents with ef={vdb_search.config.search_params['ef']}")


Test 8: Search-Time Parameters
Initializing in-memory Qdrant...
✓ In-memory Qdrant initialized
✓ Collection 'test_search_params' created
  Mode: Single-vector (384D, COSINE)
✓ Total 10 documents indexed
✓ Search parameters applied successfully
✓ Retrieved 5 documents with ef=100


In [12]:
# Test 9: Multi-Vector Mode
print("\nTest 9: Multi-Vector Mode")
print("=" * 50)

config_multi = QdrantConfig(
    db_mode="memory",
    collection_name="test_multivector",
    vectors_config={
        "text": {"dimension": 384, "distance": "COSINE"},
        "image": {"dimension": 512, "distance": "DOT"}
    }
)

vdb_multi = QdrantVectorDB(config_multi)
vdb_multi.create_collection(force_recreate=True)

# Index multi-vector documents
multi_docs = []
for i in range(5):
    multi_docs.append({
        'id': i,
        'embeddings': {
            'text': np.random.randn(384),      # Text embedding (384-D, COSINE)
            'image': np.random.randn(512)      # Image embedding (512-D, DOT)
        },
        'text': f'Multi-vector document {i}',
        'index': i
    })

vdb_multi.index_documents(multi_docs)

# Retrieve using text vector
query_text_emb = multi_docs[0]['embeddings']['text']
results_text = vdb_multi.retrieve(query_text_emb, query_vector_name="text", top_k=3)

# Retrieve using image vector
query_image_emb = multi_docs[1]['embeddings']['image']
results_image = vdb_multi.retrieve(query_image_emb, query_vector_name="image", top_k=3)

print(f"✓ Multi-vector collection created with text (384-D, COSINE) and image (512-D, DOT)")
print(f"✓ Retrieved {len(results_text)} documents using text vector")
print(f"✓ Retrieved {len(results_image)} documents using image vector")


Test 9: Multi-Vector Mode
Initializing in-memory Qdrant...
✓ In-memory Qdrant initialized
✓ Collection 'test_multivector' created
  Mode: Multi-vector with 2 vector fields
✓ Total 5 documents indexed
✓ Multi-vector collection created with text (384-D, COSINE) and image (512-D, DOT)
✓ Retrieved 3 documents using text vector
✓ Retrieved 3 documents using image vector


In [13]:
# Test 10: Multi-Vector with Different Distance Metrics
print("\nTest 10: Multi-Vector with Different Distance Metrics")
print("=" * 50)

config_multi_metrics = QdrantConfig(
    db_mode="memory",
    collection_name="test_multivector_metrics",
    vectors_config={
        "text": {"dimension": 256, "distance": "COSINE"},      # Text: COSINE
        "semantic": {"dimension": 256, "distance": "MANHATTAN"}, # Semantic: MANHATTAN
        "visual": {"dimension": 512, "distance": "EUCLID"}     # Visual: EUCLID
    }
)

vdb_multi_metrics = QdrantVectorDB(config_multi_metrics)
vdb_multi_metrics.create_collection(force_recreate=True)

# Index documents with multiple vector types
multi_metric_docs = []
for i in range(4):
    multi_metric_docs.append({
        'id': i,
        'embeddings': {
            'text': np.random.randn(256),
            'semantic': np.random.randn(256),
            'visual': np.random.randn(512)
        },
        'text': f'Multi-metric document {i}'
    })

vdb_multi_metrics.index_documents(multi_metric_docs)

# Retrieve using each vector type with its respective distance metric
results_text_cosine = vdb_multi_metrics.retrieve(
    multi_metric_docs[0]['embeddings']['text'], 
    query_vector_name="text", 
    top_k=2
)

results_semantic_manhattan = vdb_multi_metrics.retrieve(
    multi_metric_docs[1]['embeddings']['semantic'], 
    query_vector_name="semantic", 
    top_k=2
)

results_visual_euclid = vdb_multi_metrics.retrieve(
    multi_metric_docs[2]['embeddings']['visual'], 
    query_vector_name="visual", 
    top_k=2
)

print(f"✓ Multi-vector collection with 3 vector types and different metrics created:")
print(f"  - text (256-D, COSINE)")
print(f"  - semantic (256-D, MANHATTAN)")
print(f"  - visual (512-D, EUCLID)")
print(f"✓ Retrieved {len(results_text_cosine)} documents using text vector (COSINE)")
print(f"✓ Retrieved {len(results_semantic_manhattan)} documents using semantic vector (MANHATTAN)")
print(f"✓ Retrieved {len(results_visual_euclid)} documents using visual vector (EUCLID)")


Test 10: Multi-Vector with Different Distance Metrics
Initializing in-memory Qdrant...
✓ In-memory Qdrant initialized
✓ Collection 'test_multivector_metrics' created
  Mode: Multi-vector with 3 vector fields
✓ Total 4 documents indexed
✓ Multi-vector collection with 3 vector types and different metrics created:
  - text (256-D, COSINE)
  - semantic (256-D, MANHATTAN)
  - visual (512-D, EUCLID)
✓ Retrieved 2 documents using text vector (COSINE)
✓ Retrieved 2 documents using semantic vector (MANHATTAN)
✓ Retrieved 2 documents using visual vector (EUCLID)
